# ZUCE Colab: Qwen3 coding model to a 10B parameter budget

Notebook นี้ใช้ ZUCE เพื่อสกัดความสามารถด้าน coding จาก Qwen3 dense CausalLM โดยไม่ fine-tune และไม่ update weights ของ teacher

ค่าเริ่มต้นใช้ `Qwen/Qwen3-14B` แล้วตั้งงบปลายทาง `10_000_000_000` parameters. ถ้าหมายถึง `Qwen/Qwen3-8B` ให้เปลี่ยน `TEACHER_MODEL_ID` ใน cell config ได้เลย แต่ 8B จะไม่ถูกบีบให้เหลือ 10B เพราะมันเล็กกว่า budget อยู่แล้ว

แนะนำ GPU: A100 80GB หรือ H100 สำหรับ 14B -> 10B เพราะช่วง extraction ต้องโหลด teacher และสร้าง extracted model ด้วย

In [ ]:
!nvidia-smi
!pip -q install -U "git+https://github.com/YangNobody12/ZUCE.git" "transformers>=4.51.0" accelerate safetensors sentencepiece protobuf huggingface_hub

In [ ]:
# Optional: login if you use a gated/private model or want a more stable HF download session.
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
TEACHER_MODEL_ID = "Qwen/Qwen3-14B"
# If you meant the 8B model, use this instead:
# TEACHER_MODEL_ID = "Qwen/Qwen3-8B"

MAX_PARAMETERS = 10_000_000_000
OUTPUT_DIR = "/content/zuce-qwen3-coding-10b"

MAX_SAMPLES = 24
MAX_LENGTH = 512
MIN_RETENTION = 0.60
SEED = 42

# Qwen3-Coder-30B-A3B-Instruct is qwen3_moe; ZUCE v0.1 does not do MoE surgery.
# Dense Qwen3 models such as Qwen/Qwen3-8B, Qwen/Qwen3-14B, and Qwen/Qwen3-32B are the intended choices here.

In [ ]:
import json
import os
import shutil
from pathlib import Path

import torch
from transformers import AutoConfig

cfg = AutoConfig.from_pretrained(TEACHER_MODEL_ID, trust_remote_code=False)
print(json.dumps({
    "model": TEACHER_MODEL_ID,
    "model_type": cfg.model_type,
    "layers": getattr(cfg, "num_hidden_layers", None),
    "hidden_size": getattr(cfg, "hidden_size", None),
    "intermediate_size": getattr(cfg, "intermediate_size", None),
    "vocab_size": getattr(cfg, "vocab_size", None),
    "torch_dtype": str(getattr(cfg, "torch_dtype", None)),
    "cuda": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}, indent=2))

supported_dense_types = {"qwen2", "qwen3", "llama", "mistral", "gemma"}
if cfg.model_type not in supported_dense_types:
    raise RuntimeError(f"{cfg.model_type=} is not supported for physical extraction in ZUCE v0.1")
if "moe" in cfg.model_type.lower():
    raise RuntimeError("MoE models are not supported for physical extraction in ZUCE v0.1")

In [ ]:
CODING_TARGET = [
    {"messages": [{"role": "user", "content": "Write a clean Python function that validates an email address and explain the edge cases."}]},
    {"messages": [{"role": "user", "content": "Implement binary search in Python with type hints and tests."}]},
    {"messages": [{"role": "user", "content": "Refactor this loop into readable Python and explain the complexity."}]},
    {"messages": [{"role": "user", "content": "Write a FastAPI endpoint that accepts JSON, validates input, and returns errors safely."}]},
    {"messages": [{"role": "user", "content": "Debug a Python function that mutates a default list argument."}]},
    {"messages": [{"role": "user", "content": "Create a pytest test suite for a small calculator module."}]},
]

CONTRASTS = {
    "math": [
        "Solve the equation 3x + 7 = 31 step by step.",
        "Find the derivative of x^3 + 2x^2 - 5.",
        "Compute the area of a circle with radius 12.",
    ],
    "translation": [
        "Translate this sentence into Thai: The weather is beautiful today.",
        "Translate this paragraph into English: ฉันกำลังเรียนรู้การเขียนโปรแกรม",
    ],
    "general": [
        "Summarize the causes of seasonal weather changes.",
        "Explain why regular exercise is useful for health.",
    ],
}

len(CODING_TARGET), {name: len(value) for name, value in CONTRASTS.items()}

In [ ]:
from zuce import CapabilitySpec, ParameterBudget, ZUCE

output_path = Path(OUTPUT_DIR)
if output_path.exists():
    raise FileExistsError(f"Output already exists: {output_path}. Change OUTPUT_DIR or remove it manually.")

result = ZUCE.extract(
    model=TEACHER_MODEL_ID,
    capability=CapabilitySpec(
        name="coding",
        target=CODING_TARGET,
        contrasts=CONTRASTS,
    ),
    budget=ParameterBudget(max_parameters=MAX_PARAMETERS),
    output_dir=OUTPUT_DIR,
    device="auto",
    dtype="bfloat16",
    trust_remote_code=False,
    max_samples=MAX_SAMPLES,
    max_length=MAX_LENGTH,
    min_retention=MIN_RETENTION,
    seed=SEED,
)

print(json.dumps(result.to_dict(), indent=2))

In [ ]:
proof = ZUCE.verify(OUTPUT_DIR)
print(json.dumps(proof, indent=2))

for filename in ["zuce_manifest.json", "zero_update_proof.json", "evaluation_report.json"]:
    path = Path(OUTPUT_DIR) / filename
    print("\n==", filename, "==")
    print(path.read_text(encoding="utf-8")[:4000])

In [ ]:
# Quick smoke test: load the extracted HF model and ask for code.
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR, trust_remote_code=False)
model = AutoModelForCausalLM.from_pretrained(
    OUTPUT_DIR,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=False,
)
model.eval()

messages = [{"role": "user", "content": "Write a Python function `chunked(iterable, size)` with a short pytest test."}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

with torch.no_grad():
    generated = model.generate(**inputs, max_new_tokens=256, temperature=0.2, do_sample=True)

print(tokenizer.decode(generated[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True))

In [ ]:
# Package the artifact for download or Drive upload.
archive = shutil.make_archive(OUTPUT_DIR, "zip", OUTPUT_DIR)
print("Created", archive)

# Optional Google Drive copy:
# from google.colab import drive
# drive.mount('/content/drive')
# shutil.copy2(archive, '/content/drive/MyDrive/')